# NDP Endpoint Tutorial: S3 Storage Management

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/national-data-platform/ep-api/blob/main/docs/s3_api_tutorial.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/national-data-platform/ep-api/main?filepath=docs/s3_api_tutorial.ipynb)

This notebook uses the S3 routes of an NDP Endpoint to manage
S3-compatible object storage (the bundled MinIO, or any S3 service the
Endpoint is configured for). You will:

1. Create and inspect a bucket
2. Upload, list, inspect and download objects
3. Generate presigned download and upload URLs
4. See the errors the API returns
5. Delete the objects and the bucket

The bucket name carries a unique run id; the bucket and everything in
it are deleted at the end.

## Prerequisites

- Python 3.11+ and the `requests` library
- An NDP Endpoint started with `S3_ENABLED=True` (the S3 routes only
  exist then) and a token that may use them

## Routes used

| Route | Method | Purpose |
|-------|--------|---------|
| `/s3/buckets/` | GET / POST | List / create buckets (note the trailing slash) |
| `/s3/buckets/{bucket}` | GET / DELETE | Bucket info / delete an empty bucket |
| `/s3/objects/{bucket}` | GET / POST | List objects (`?prefix=`) / upload (multipart `file` + form field `object_key`) |
| `/s3/objects/{bucket}/{key}` | GET / DELETE | Download / delete an object |
| `/s3/objects/{bucket}/{key}/metadata` | GET | Object metadata |
| `/s3/objects/{bucket}/{key}/presigned-download` | POST | Temporary download URL (`{"expires_in": seconds}`) |
| `/s3/objects/{bucket}/{key}/presigned-upload` | POST | Temporary upload URL (`{"expires_in": seconds}`) |

In [ ]:
# Install the only dependency (already present on most Jupyter setups)
%pip install -q requests

## 1. Configuration

The notebook reads two environment variables so it can run unchanged
against any Endpoint:

| Variable | Default | Meaning |
|----------|---------|---------|
| `EP_API_URL` | `http://localhost:8002` | Base URL of the Endpoint API (8002 is the port published by the bundled `docker-compose.yml`). |
| `EP_TOKEN` | `testing_token` | Bearer token sent in the `Authorization` header. |

> **About the token:** `testing_token` is the development `TEST_TOKEN`
> that a local Endpoint accepts when it is started with `TEST_TOKEN=testing_token`.
> A real Endpoint does not accept it: export `EP_TOKEN` with your NDP
> access token instead (write operations need a token whose user may
> write on this Endpoint).

In [ ]:
import datetime
import json
import os
import uuid

import requests

API_BASE_URL = os.getenv("EP_API_URL", "http://localhost:8002").rstrip("/")
TOKEN = os.getenv("EP_TOKEN", "testing_token")
HEADERS = {"Authorization": f"Bearer {TOKEN}"}

# A unique suffix for everything this notebook creates, so it never
# collides with existing data and can be cleaned up precisely.
RUN_ID = (
    datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d%H%M%S")
    + uuid.uuid4().hex[:6]
)

print(f"API base URL: {API_BASE_URL}")
print(f"Run id:       {RUN_ID}")

### Helper functions

In [ ]:
def call(method, path, expected=(200, 201), **kwargs):
    """
    Send an authenticated request to the Endpoint and return (status, body).

    Raises RuntimeError when the status code is not in `expected`, so a
    failing step stops the notebook instead of silently cascading.
    Pass `expected=None` to accept any status (used for error demos).
    """
    headers = {**HEADERS, **kwargs.pop("headers", {})}
    kwargs.setdefault("timeout", 60)
    response = requests.request(
        method, f"{API_BASE_URL}{path}", headers=headers, **kwargs
    )
    try:
        body = response.json()
    except ValueError:
        body = response.text
    print(f"{method} {path} -> {response.status_code}")
    if expected is not None and response.status_code not in expected:
        raise RuntimeError(
            f"Unexpected status {response.status_code}: {json.dumps(body, indent=2)}"
        )
    return response.status_code, body


def show(data):
    """Pretty-print a JSON-like value."""
    print(json.dumps(data, indent=2, default=str))

## 2. Check the Endpoint

`GET /status/` (note the trailing slash) needs a token and reports
the API version and which optional features this Endpoint has
enabled.

In [ ]:
_, status = call("GET", "/status/")
for key in ("api_version", "s3_enabled", "s3_connected"):
    print(f"{key:24} {status.get(key)}")

In [ ]:
if not (status.get("s3_enabled") and status.get("s3_connected")):
    raise RuntimeError("S3 is not enabled or not reachable on this Endpoint.")

## 3. Buckets

List the existing buckets, then create one for this tutorial. Bucket
names follow the S3 rules: 3 to 63 lowercase letters, digits and `-`.

In [ ]:
_, listing = call("GET", "/s3/buckets/")
print(f"{len(listing['buckets'])} buckets: {[b['name'] for b in listing['buckets']][:10]}")

In [ ]:
BUCKET = f"tutorial-s3-{RUN_ID}"

_, created = call("POST", "/s3/buckets/", json={"name": BUCKET})
show(created)

_, info = call("GET", f"/s3/buckets/{BUCKET}")
show(info)

## 4. Objects

### Upload

`POST /s3/objects/{bucket}` is a multipart upload: the file goes in the
`file` part and the target key in the optional `object_key` form field
(the file name is used when it is omitted). Keys may contain `/` to
organise objects in folders. The response has `bucket`, `key`, `size`
and `etag`.

In [ ]:
sample_files = {
    "data/sample_data.txt": (
        "text/plain",
        "Sample data file for the S3 tutorial.\ntemperature,humidity\n21.5,40\n",
    ),
    "data/config.json": (
        "application/json",
        json.dumps({"tutorial": "s3", "run_id": RUN_ID}, indent=2),
    ),
    "README.md": ("text/markdown", "# S3 tutorial\n\nFiles uploaded by the S3 tutorial.\n"),
}

for key, (content_type, content) in sample_files.items():
    _, uploaded = call(
        "POST",
        f"/s3/objects/{BUCKET}",
        files={"file": (key.rsplit("/", 1)[-1], content.encode(), content_type)},
        data={"object_key": key},
    )
    print(f"  key={uploaded['key']} size={uploaded['size']} etag={uploaded['etag']}")

### List

`GET /s3/objects/{bucket}` lists every object; `?prefix=` narrows the
listing to keys that start with a given prefix.

In [ ]:
_, objects = call("GET", f"/s3/objects/{BUCKET}")
for obj in objects["objects"]:
    print(f"  {obj['key']:24} {obj['size']:>5} bytes  modified {obj['last_modified']}")

_, in_data = call("GET", f"/s3/objects/{BUCKET}", params={"prefix": "data/"})
print("with prefix 'data/':", [o["key"] for o in in_data["objects"]])

### Metadata

In [ ]:
_, metadata = call("GET", f"/s3/objects/{BUCKET}/data/sample_data.txt/metadata")
for field in ("key", "size", "content_type", "last_modified", "etag"):
    print(f"  {field:14} {metadata.get(field)}")

### Download

`GET /s3/objects/{bucket}/{key}` streams the object back through the
Endpoint, with a `Content-Disposition: attachment` header.

In [ ]:
response = requests.get(
    f"{API_BASE_URL}/s3/objects/{BUCKET}/data/sample_data.txt", headers=HEADERS, timeout=60
)
response.raise_for_status()
print("Content-Type:       ", response.headers.get("content-type"))
print("Content-Disposition:", response.headers.get("content-disposition"))
print("matches the upload: ", response.text == sample_files["data/sample_data.txt"][1])
print(response.text)

## 5. Presigned URLs

A presigned URL lets someone without a token download (or upload) one
object for a limited time. Request one with `POST` and a JSON body
`{"expires_in": seconds}` (default 3600, maximum 7 days).

> **Where presigned URLs work:** the URL is signed by the Endpoint for
> the S3 address it is configured with (`S3_ENDPOINT`). With the bundled
> `docker-compose.yml` that is `minio:9000`, a host name that only
> resolves inside the compose network. The signature covers the host,
> so rewriting it to the published MinIO port (`localhost:9002`) does
> not work either. Configure `S3_ENDPOINT` with an address your clients
> can reach if you want to hand presigned URLs out. The cells below
> generate the URLs and try them, and report when the URL is not
> reachable from where the notebook runs.

In [ ]:
def try_presigned(method, url, **kwargs):
    try:
        response = requests.request(method, url, timeout=10, **kwargs)
        print(f"  {method} presigned URL -> {response.status_code}")
        return response
    except requests.exceptions.RequestException as exc:
        host = url.split("/")[2]
        print(f"  {host} is not reachable from here ({type(exc).__name__}); "
              "see the note above.")
        return None


_, presigned = call(
    "POST", f"/s3/objects/{BUCKET}/data/sample_data.txt/presigned-download", json={"expires_in": 600}
)
print("expires in:", presigned["expires_in"], "seconds")
print("url:", presigned["url"][:90], "...")

response = try_presigned("GET", presigned["url"])
if response is not None and response.ok:
    print("  downloaded:", response.text[:60])

In [ ]:
PRESIGNED_KEY = "uploads/presigned_upload.txt"

_, presigned_upload = call(
    "POST", f"/s3/objects/{BUCKET}/{PRESIGNED_KEY}/presigned-upload", json={"expires_in": 600}
)
print("url:", presigned_upload["url"][:90], "...")

# The client uploads with a plain HTTP PUT, no token needed
try_presigned(
    "PUT",
    presigned_upload["url"],
    data=b"Uploaded with a presigned URL.\n",
    headers={"Content-Type": "text/plain"},
)

## 6. Errors

The API answers with a JSON body whose `detail` explains the problem:

In [ ]:
error_cases = [
    ("missing bucket", "GET", f"/s3/objects/no-such-bucket-{RUN_ID}", {}),
    ("missing object", "GET", f"/s3/objects/{BUCKET}/no-such-file.txt/metadata", {}),
    ("invalid bucket name", "POST", "/s3/buckets/", {"json": {"name": "Invalid Name!"}}),
    ("bucket already exists", "POST", "/s3/buckets/", {"json": {"name": BUCKET}}),
    ("bucket not empty", "DELETE", f"/s3/buckets/{BUCKET}", {}),
]

for label, method, path, kwargs in error_cases:
    status_code, body = call(method, path, expected=None, **kwargs)
    print(f"  {label}: {status_code} {body.get('detail')}\n")

## 7. Uploading several files

In [ ]:
def upload_many(bucket, files):
    results = {}
    for key, content in files.items():
        status_code, body = call(
            "POST",
            f"/s3/objects/{bucket}",
            expected=None,
            files={"file": (key.rsplit("/", 1)[-1], content.encode(), "text/plain")},
            data={"object_key": key},
        )
        results[key] = status_code in (200, 201)
    return results


batch = {f"batch/file_{i}.txt": f"Batch file {i}\n" for i in range(1, 4)}
results = upload_many(BUCKET, batch)
print(f"uploaded {sum(results.values())}/{len(results)}")

## 8. Clean up

A bucket must be empty before it can be deleted, so delete every
object first (`DELETE /s3/objects/{bucket}/{key}`), then the bucket.

In [ ]:
_, objects = call("GET", f"/s3/objects/{BUCKET}")
for obj in objects["objects"]:
    call("DELETE", f"/s3/objects/{BUCKET}/{obj['key']}")

call("DELETE", f"/s3/buckets/{BUCKET}")

_, listing = call("GET", "/s3/buckets/")
left = BUCKET in [b["name"] for b in listing["buckets"]]
print("bucket left:", left)
assert not left

## Summary

You created a bucket, uploaded, listed, inspected and downloaded
objects, generated presigned URLs and removed everything again.

The [S3 to dataset workflow](./s3_to_dataset_workflow_tutorial.ipynb)
shows how to register stored objects in the catalog so others can
find them. More in the
[ep-api repository](https://github.com/national-data-platform/ep-api).